# SageMaker batch transform - MedGemma

Uploads a MedGemma JSONL request batch under the S3 input prefix allowed by the Marketplace batch CloudFormation template, validates the CloudFormation-created model and execution role, starts a SageMaker batch transform job, waits for completion, reads the output, and optionally removes test S3 objects.

## Credentials

Use the notebook instance/profile role, or set `AWS_PROFILE` below when running locally. Avoid storing static access keys in the notebook.

In [ ]:
import base64
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint

from botocore.exceptions import ClientError
from boto3.session import Session

## Config & clients

In [ ]:
REGION_NAME = "us-east-1"
AWS_PROFILE = None  # Example: "sandbox". Leave None to use the current notebook/instance credentials.

MODEL_NAME = "your-cloudformation-model-name-output"
EXECUTION_ROLE_ARN = "your-cloudformation-execution-role-arn-output"
DATA_BUCKET_NAME = "your-cloudformation-batch-data-bucket-output"
BATCH_INPUT_PREFIX = "input/"   # Must match the CF BatchInputS3Prefix parameter/output.
BATCH_OUTPUT_PREFIX = "output/" # Must match the CF BatchOutputS3Prefix parameter/output.

MODEL_ID = "google/medgemma-1.5-4b-it"
BATCH_INSTANCE_TYPE = "ml.g5.2xlarge"
BATCH_INSTANCE_COUNT = 1
BATCH_TRANSFORM_AMI_VERSION = "al2-ami-sagemaker-batch-gpu-535"

def child_prefix(base_prefix: str, child: str) -> str:
    return f"{base_prefix.strip('/')}/{child.strip('/')}/"


RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
BATCH_TRANSFORM_JOB_NAME = f"medgemma-marketplace-batch-{RUN_ID}"
INPUT_PREFIX = child_prefix(BATCH_INPUT_PREFIX, RUN_ID)
OUTPUT_PREFIX = child_prefix(BATCH_OUTPUT_PREFIX, RUN_ID)

session_kwargs = {"region_name": REGION_NAME}
if AWS_PROFILE:
    session_kwargs["profile_name"] = AWS_PROFILE

session = Session(**session_kwargs)
s3 = session.client("s3")
sagemaker = session.client("sagemaker")
logs = session.client("logs")

print(f"Run id: {RUN_ID}")
print(f"Model:  {MODEL_NAME}")
print(f"Role:   {EXECUTION_ROLE_ARN}")
print(f"Input:  s3://{DATA_BUCKET_NAME}/{INPUT_PREFIX}")
print(f"Output: s3://{DATA_BUCKET_NAME}/{OUTPUT_PREFIX}")

In [ ]:
assert MODEL_NAME != "your-cloudformation-model-name-output", "Set MODEL_NAME from the CloudFormation ModelName output before running."
assert EXECUTION_ROLE_ARN != "your-cloudformation-execution-role-arn-output", "Set EXECUTION_ROLE_ARN from the CloudFormation ExecutionRoleArn output before running."
assert DATA_BUCKET_NAME != "your-cloudformation-batch-data-bucket-output", "Set DATA_BUCKET_NAME from the CloudFormation BatchDataBucketName output before running."
assert BATCH_INPUT_PREFIX.strip("/"), "Set BATCH_INPUT_PREFIX to the CloudFormation BatchInputS3Prefix value."
assert BATCH_OUTPUT_PREFIX.strip("/"), "Set BATCH_OUTPUT_PREFIX to the CloudFormation BatchOutputS3Prefix value."
assert INPUT_PREFIX.startswith(BATCH_INPUT_PREFIX.strip("/") + "/")
assert OUTPUT_PREFIX.startswith(BATCH_OUTPUT_PREFIX.strip("/") + "/")
assert len(BATCH_TRANSFORM_JOB_NAME) <= 63

## Locate input image

In [ ]:
def find_repo_file(*parts: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base.joinpath(*parts)
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {'/'.join(parts)} from {Path.cwd()}")


def find_input_image() -> Path:
    candidates = [
        (Path.cwd() / "inputs" / "chest_xray.png"),
        (Path.cwd() / "tests" / "inputs" / "chest_xray.png"),
        (Path.cwd().parent / "inputs" / "chest_xray.png"),
        find_repo_file("tests", "inputs", "chest_xray.png"),
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError("Could not find chest_xray.png from this notebook working directory")


IMAGE_PATH = find_input_image()
image_b64 = base64.b64encode(IMAGE_PATH.read_bytes()).decode("ascii")

print(IMAGE_PATH)
print(f"Payload image size: {len(image_b64) / 1024 / 1024:.2f} MiB base64")

## Build and upload JSONL request batch

In [ ]:
def build_payload(prompt: str, max_tokens: int = 512) -> dict:
    return {
        "model": MODEL_ID,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{image_b64}"},
                    },
                ],
            },
        ],
        "max_tokens": max_tokens,
        "temperature": 0,
    }


PROMPTS = [
    "Describe this chest X-ray. Note any abnormal findings.",
    "Summarize the most important radiographic observations in one paragraph.",
    "List possible findings and explicitly mention uncertainty.",
    "Does this image show any obvious acute cardiopulmonary abnormality? Explain briefly.",
    "Create a concise impression section for this chest X-ray.",
]

records = [build_payload(prompt) for prompt in PROMPTS]
jsonl_body = "\n".join(json.dumps(record) for record in records) + "\n"
jsonl_bytes = jsonl_body.encode("utf-8")
input_key = f"{INPUT_PREFIX}requests.jsonl"

s3.put_object(
    Bucket=DATA_BUCKET_NAME,
    Key=input_key,
    Body=jsonl_bytes,
    ContentType="application/jsonlines",
)

print(f"Uploaded {len(records)} JSONL records to s3://{DATA_BUCKET_NAME}/{input_key}")
print(f"JSONL object size: {len(jsonl_bytes) / 1024 / 1024:.2f} MiB")

## Validate CloudFormation-created model

In [ ]:
model = sagemaker.describe_model(ModelName=MODEL_NAME)
model_execution_role = model.get("ExecutionRoleArn")
assert model_execution_role == EXECUTION_ROLE_ARN, (
    "The SageMaker model is not using the expected execution role. "
    f"Expected {EXECUTION_ROLE_ARN}, got {model_execution_role}"
)

pprint({
    "ModelName": model["ModelName"],
    "ModelArn": model["ModelArn"],
    "ExecutionRoleArn": model_execution_role,
    "EnableNetworkIsolation": model.get("EnableNetworkIsolation"),
    "PrimaryContainer": model.get("PrimaryContainer"),
})

## Start the transform job directly

In [ ]:
def start_transform_job() -> dict:
    try:
        existing = sagemaker.describe_transform_job(TransformJobName=BATCH_TRANSFORM_JOB_NAME)
        print(f"Transform job already exists: {existing['TransformJobStatus']}")
        return existing
    except ClientError as exc:
        if "Could not find" not in str(exc) and "ResourceNotFound" not in str(exc):
            raise

    resources = {
        "InstanceType": BATCH_INSTANCE_TYPE,
        "InstanceCount": BATCH_INSTANCE_COUNT,
    }
    if BATCH_TRANSFORM_AMI_VERSION:
        resources["TransformAmiVersion"] = BATCH_TRANSFORM_AMI_VERSION

    sagemaker.create_transform_job(
        TransformJobName=BATCH_TRANSFORM_JOB_NAME,
        ModelName=MODEL_NAME,
        MaxConcurrentTransforms=1,
        MaxPayloadInMB=6,
        BatchStrategy="SingleRecord",
        TransformInput={
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{DATA_BUCKET_NAME}/{INPUT_PREFIX}",
                }
            },
            "ContentType": "application/json",
            "SplitType": "Line",
            "CompressionType": "None",
        },
        TransformOutput={
            "S3OutputPath": f"s3://{DATA_BUCKET_NAME}/{OUTPUT_PREFIX}",
            "Accept": "application/json",
            "AssembleWith": "Line",
        },
        TransformResources=resources,
    )
    return sagemaker.describe_transform_job(TransformJobName=BATCH_TRANSFORM_JOB_NAME)


job = start_transform_job()
pprint({
    "TransformJobName": job["TransformJobName"],
    "TransformJobStatus": job["TransformJobStatus"],
    "Input": job.get("TransformInput"),
    "Output": job.get("TransformOutput"),
})

## Watch the transform job

In [ ]:
def describe_transform_job() -> dict:
    return sagemaker.describe_transform_job(TransformJobName=BATCH_TRANSFORM_JOB_NAME)


def wait_for_transform_job(poll_seconds: int = 30, max_minutes: int = 90) -> dict:
    terminal = {"Completed", "Failed", "Stopped"}
    deadline = time.time() + max_minutes * 60
    while True:
        job = describe_transform_job()
        status = job["TransformJobStatus"]
        print(datetime.now().strftime("%H:%M:%S"), status, job.get("FailureReason", ""))
        if status in terminal:
            return job
        if time.time() > deadline:
            raise TimeoutError(f"Timed out waiting for {BATCH_TRANSFORM_JOB_NAME}")
        time.sleep(poll_seconds)


job = wait_for_transform_job()
pprint({
    "Status": job["TransformJobStatus"],
    "FailureReason": job.get("FailureReason"),
    "Input": job.get("TransformInput"),
    "Output": job.get("TransformOutput"),
})

## Logs

In [ ]:
def list_transform_log_streams() -> list[dict]:
    try:
        response = logs.describe_log_streams(
            logGroupName="/aws/sagemaker/TransformJobs",
            logStreamNamePrefix=f"{BATCH_TRANSFORM_JOB_NAME}/",
            orderBy="LastEventTime",
            descending=True,
            limit=20,
        )
    except logs.exceptions.ResourceNotFoundException:
        return []
    return response.get("logStreams", [])


streams = list_transform_log_streams()
pprint(streams)

for stream in streams[:3]:
    name = stream["logStreamName"]
    events = logs.get_log_events(
        logGroupName="/aws/sagemaker/TransformJobs",
        logStreamName=name,
        startFromHead=False,
        limit=20,
    ).get("events", [])
    print(f"\n--- {name} ---")
    for event in events[-20:]:
        print(event.get("message", ""))

## Read output

In [ ]:
def list_s3_objects(bucket: str, prefix: str) -> list[dict]:
    paginator = s3.get_paginator("list_objects_v2")
    objects = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        objects.extend(page.get("Contents", []))
    return objects


def parse_json_line(line: str):
    try:
        return json.loads(line)
    except json.JSONDecodeError:
        return line


objects = list_s3_objects(DATA_BUCKET_NAME, OUTPUT_PREFIX)
if not objects:
    print(f"No output objects found under s3://{DATA_BUCKET_NAME}/{OUTPUT_PREFIX}")
else:
    for obj in objects:
        key = obj["Key"]
        body = s3.get_object(Bucket=DATA_BUCKET_NAME, Key=key)["Body"].read().decode("utf-8")
        lines = [line for line in body.splitlines() if line.strip()]
        print(f"\n--- s3://{DATA_BUCKET_NAME}/{key} ({len(lines)} line(s)) ---")
        for idx, line in enumerate(lines, start=1):
            print(f"\n[{idx}]")
            parsed = parse_json_line(line)
            pprint(parsed if not isinstance(parsed, str) else parsed[:4000])

## Cleanup

In [ ]:
DELETE_S3_TEST_OBJECTS = False
STOP_TRANSFORM_JOB_IF_RUNNING = False


def list_s3_keys(bucket: str, prefix: str) -> list[str]:
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        keys.extend(obj["Key"] for obj in page.get("Contents", []))
    return keys


if STOP_TRANSFORM_JOB_IF_RUNNING:
    try:
        current = sagemaker.describe_transform_job(TransformJobName=BATCH_TRANSFORM_JOB_NAME)
        if current["TransformJobStatus"] in {"InProgress", "Stopping"}:
            sagemaker.stop_transform_job(TransformJobName=BATCH_TRANSFORM_JOB_NAME)
            print(f"Stopping transform job: {BATCH_TRANSFORM_JOB_NAME}")
    except ClientError as exc:
        print(f"Skipping transform stop: {exc}")

if DELETE_S3_TEST_OBJECTS:
    keys = list_s3_keys(DATA_BUCKET_NAME, INPUT_PREFIX) + list_s3_keys(DATA_BUCKET_NAME, OUTPUT_PREFIX)
    for i in range(0, len(keys), 1000):
        batch = keys[i:i + 1000]
        if batch:
            s3.delete_objects(
                Bucket=DATA_BUCKET_NAME,
                Delete={"Objects": [{"Key": key} for key in batch], "Quiet": True},
            )
    print(f"Deleted {len(keys)} S3 test object(s).")